# 🐍 Notebook 07: Python Functions, *args, **kwargs, Closures & Lambdas
**Trainer: Sundeep Hiremat · TASK Nalgonda Center**  
**Course:** Cloud Software Developer (AWS DVA-C02 Aligned)

In this notebook, we follow the **KISS (Keep It Simple, Student)** principle to master core Python functions:
1. **Function Fundamentals & Defaults:** Defining clean functions, default parameters, and avoiding mutable default traps.
2. **Variable Arguments (`*args` and `**kwargs`):** Handling dynamic arguments (tuples and dictionaries) as seen in AWS SDKs (`boto3`).
3. **Closures (Functions inside Functions):** How inner functions capture and remember enclosing scope variables.
4. **Lambda Functions (Anonymous Functions):** Quick, inline logic for sorting, filtering, and mapping.


## 1. Functions & Dynamic Arguments (`*args`, `**kwargs`)
In AWS development, functions often need to accept dynamic options (like tags, metadata, or pagination parameters).


In [ ]:
# 1. Standard Function with Default & Type Hints
def calculate_instance_cost(hours: int, rate_per_hour: float = 0.0208) -> float:
    """Calculates total run cost for an EC2 t3.micro instance."""
    total = hours * rate_per_hour
    return round(total, 4)

print("Cost for 24 hours:", calculate_instance_cost(24))
print("Cost for 720 hours (1 month):", calculate_instance_cost(720))


### Variable Arguments: `*args` (Tuple) and `**kwargs` (Dictionary)
* `*args`: Collects extra positional arguments into a **tuple**.
* `**kwargs`: Collects extra keyword arguments into a **dictionary**.


In [ ]:
def deploy_cloud_service(service_name, region="ap-south-1", *tags, **metadata):
    print(f"🚀 Deploying: {service_name} in {region}")
    print(f"📌 Positional Tags (tuple): {tags}")
    print(f"⚙️ Metadata Options (dict): {metadata}")

# Calling with variable arguments
deploy_cloud_service(
    "Amazon EC2",
    "us-east-1",
    "Production", "WebTier", "Nalgonda",
    Owner="Sundeep", InstanceType="t3.medium", AutoTerminate=True
)


---
## 2. Lambda Functions (Anonymous Functions)
A **lambda function** is a small, one-line function without a name.  
$$\text{Syntax: } \text{lambda } arguments : expression$$

> 💡 **Notice:** This is a Python language feature! Don't confuse it with **AWS Lambda** (the cloud compute service).


In [ ]:
# Regular function vs Lambda
def add_tax(price):
    return price * 1.18

# Same logic in a lambda
add_tax_lambda = lambda price: price * 1.18

print("Regular function:", add_tax(100))
print("Lambda function:", add_tax_lambda(100))


### Real Cloud Use Case: Sorting Cloud Fleets by Cost
In cloud engineering, we frequently fetch lists of servers from AWS and sort them by price, CPU, or launch time:


In [ ]:
# List of running EC2 instances
ec2_inventory = [
    {"instance_id": "i-001", "type": "t3.large", "hourly_usd": 0.0832, "state": "running"},
    {"instance_id": "i-002", "type": "t2.nano",  "hourly_usd": 0.0058, "state": "stopped"},
    {"instance_id": "i-003", "type": "t3.micro", "hourly_usd": 0.0208, "state": "running"},
    {"instance_id": "i-004", "type": "c5.xlarge", "hourly_usd": 0.1700, "state": "running"}
]

# 1. Sort by hourly cost ascending using a lambda
sorted_by_cost = sorted(ec2_inventory, key=lambda inst: inst["hourly_usd"])

print("EC2 Fleet sorted by cost (cheapest first):")
for inst in sorted_by_cost:
    print(f"  • {inst['instance_id']} ({inst['type']}): ${inst['hourly_usd']}/hr")


### Filtering with Lambdas (`filter` and List Comprehensions)


In [ ]:
# Filter only RUNNING instances using lambda
running_instances = list(filter(lambda inst: inst["state"] == "running", ec2_inventory))
print("Running instances:", [i["instance_id"] for i in running_instances])

# Equivalent list comprehension (very idiomatic in Python)
running_comp = [i["instance_id"] for i in ec2_inventory if i["state"] == "running"]
print("Using list comprehension:", running_comp)


---
## 3. Function Closures (Simple & Elegant)

### 🧠 What is a Closure?
> **Analogy for Students:** Imagine a personalized stamp maker. You tell the factory once: *'Make me a stamp with the prefix TASK-Nalgonda'*. The factory hands you a small tool. Whenever you press that tool on a student ID, it remembers the prefix forever without you having to re-type it!

A **closure** is simply an **inner function** that remembers the variables from its **enclosing (outer) scope**, even after the outer function has finished executing.


In [ ]:
# 1. Outer function creates and returns an inner function
def make_cost_calculator(hourly_rate, currency="USD"):
    # Outer scope variable 'hourly_rate' and 'currency' are remembered
    def calculate(hours):
        total = hours * hourly_rate
        return f"{total:.2f} {currency}"
    return calculate

# 2. Creating specialized calculator functions
t3_micro_calc = make_cost_calculator(0.0208, "USD")
t3_medium_calc = make_cost_calculator(0.0416, "USD")

# 3. Calling the closure with just the hours parameter
print("t3.micro for 24 hours:  ", t3_micro_calc(24))
print("t3.micro for 720 hours: ", t3_micro_calc(720))
print("t3.medium for 24 hours: ", t3_medium_calc(24))


---
## 4. Key Takeaways (KISS Principle)
| Feature | Syntax | Behavior | When to Use |
| :--- | :--- | :--- | :--- |
| **Standard Function** | `def func(x, y=10):` | Named, reusable block with optional defaults | Core business logic |
| **Positional Sack (`*args`)** | `def func(*args):` | Collects arbitrary extra arguments into a **tuple** | Unknown number of inputs (e.g. email lists) |
| **Keyword Tags (`**kwargs`)** | `def func(**kwargs):` | Collects arbitrary extra named arguments into a **dict** | Optional cloud configs, tags, metadata |
| **Closures** | `def outer(x): def inner():` | Inner function retains outer scope state | Reusable factory functions & multipliers |
| **Lambdas** | `lambda x: x * 2` | 1-line anonymous disposable expression | Quick sorting keys (`sorted()`) and filters |
